# 03 - Demand Forecasting Engine

**Project:** E-Commerce Dynamic Pricing & Demand Forecasting Engine (Olist Brazil)
**Stage:** Stage 2 | **Phase:** 3 - Demand Forecasting Engine
**Source modules:** `src/models/*`, `src/features/*`, `src/data/*`, `src/utils/*`
**Certified by:** `tests/test_features.py`, `tests/test_forecasting_models.py` (+ prior suites)

Workflow per PROJECT_BLUEPRINT.md Part 3:

1. **Phase-3 feature stack** (lags 1/2/4, rolling mean/std, calendar, drivers).
2. **Fold-boundary equivalence proof** - recomputing features inside a
   chronological window yields identical values to the global past-only pass.
3. **Chronological 70/15/15 split + train-only imputation.**
4. **Expanding temporal CV** + Optuna tuning (LightGBM, XGBoost) on train only.
5. **Full-data final fits** with early stopping carved from the train tail.
6. **MLflow lifecycle** - every candidate logged; winner packaged as a validated
   PyFunc model with an explicit signature contract.

> Test split stays untouched until the Phase-5 end-to-end evaluation.


## 1. Setup

In [1]:
import sys
from pathlib import Path

_cwd = Path.cwd()
def _find_root(path: Path) -> Path:
    while path != path.parent:
        if (path / "src").is_dir() and (path / "data").is_dir():
            return path
        path = path.parent
    return path

ROOT = _find_root(_cwd)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import gc
import json

import numpy as np
import pandas as pd
import mlflow

from src.utils.logging_setup import get_logger
from src.utils.metrics import describe_metrics
from src.data.target_engineering import load_demand_matrix
from src.features.feature_engineering import (
    build_phase3_feature_matrix,
    restrict_to_window,
)
from src.data.temporal_split import temporal_split, assert_chronological_order
from src.data.imputation import impute_splits
from src.models.data_prep import prepare_model_matrix, design_matrix_rows
from src.models.forecasting import (
    make_model,
    naive_combined_predictions,
    expanding_window_splits,
    run_expanding_cv,
    fit_full_with_early_stopping,
)
from src.models.optuna_tuning import tune_hyperparameters
from src.models.pyfunc_wrapper import save_winner

LOGGER = get_logger("notebook_03")
ARTIFACTS_DIR = ROOT / "artifacts" / "phase3"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'artifacts' / 'mlflow_tracking.db'}")
mlflow.set_experiment("demand_forecasting_phase3")
LOGGER.info("MLflow tracking -> %s", mlflow.get_tracking_uri())

C:\Users\Hedaya_city\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Hedaya_city\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


2026/09/22 00:17:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas


2026/09/22 00:17:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables


2026/09/22 00:17:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types


...

2026/09/22 00:17:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults


2026/09/22 00:17:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments


2026/09/22 00:17:49 INFO alembic.runtime.migration: Context impl SQLiteImpl.


2026/09/22 00:17:49 INFO alembic.runtime.migration: Will assume non-transactional DDL.


2026-09-22 00:17:49 | INFO    | notebook_03 | MLflow tracking -> sqlite:///D:\ALL Projects\E-Commerce\artifacts\mlflow_tracking.db


## 2. Load Demand Matrix & Build Phase-3 Features

Features are computed on the full chronological matrix. Each feature is strictly
past-only (lags via ``shift``, rolling on a shift-by-1 series, drivers lagged),
so no row ever carries information from its own week or later weeks.

In [2]:
matrix = load_demand_matrix(ROOT / "data" / "processed" / "demand_matrix_weekly.csv.gz")
LOGGER.info("matrix loaded: %d rows, weeks %s..%s", len(matrix), matrix["week"].min(), matrix["week"].max())

features_full = build_phase3_feature_matrix(matrix)
del matrix
gc.collect()
print(f"Phase-3 feature frame: {features_full.shape}")

2026-09-22 00:18:25 | INFO    | src.data.target_engineering | load_demand_matrix: loaded 3469274 rows x 6 cols.


2026-09-22 00:18:27 | INFO    | notebook_03 | matrix loaded: 3469274 rows, weeks 2016-08-29/2016-09-04..2018-09-03/2018-09-09


2026-09-22 00:18:27 | INFO    | src.features.feature_engineering | build_phase3_feature_matrix: starting (3469274 rows).


2026-09-22 00:18:35 | INFO    | src.features.feature_engineering | add_demand_drivers: price_prev, price_chg_1w, freight_prev, category_share_prev added.


2026-09-22 00:18:38 | INFO    | src.features.feature_engineering | add_lag_features: lags (1, 2, 4) added.


2026-09-22 00:18:50 | INFO    | src.features.feature_engineering | add_rolling_features: windows (4,) aggs ('mean', 'std') added (shifted-by-1, past-only): ['rolling_demand_mean_4w', 'rolling_demand_std_4w']


2026-09-22 00:18:51 | INFO    | src.features.feature_engineering | build_phase3_feature_matrix: completed -> 3469274 x 20


Phase-3 feature frame: (3469274, 20)


## 3. Fold-Boundary Equivalence Proof

Blueprint rule: *"feature engineering strictly on training-set boundaries"*.
Because every feature is past-only, recomputing on a window truncated at week
`B` must produce **identical** values to the global pass for all rows `<= B`.
This proof runs on mature products for speed; the same structural guarantee is
enforced behaviorally by the test suite's leakage probe.

In [3]:
mature = (
    features_full.groupby("product_id")["order_demand"].sum()
    .sort_values(ascending=False).head(150).index
)
subset_matrix = load_demand_matrix(ROOT / "data" / "processed" / "demand_matrix_weekly.csv.gz")[
    lambda d: d["product_id"].isin(mature)
]
global_features = build_phase3_feature_matrix(subset_matrix).set_index(["product_id", "week"])
interior = len(subset_matrix["week"].unique()) // 2
cutoff = sorted(subset_matrix["week"].unique())[interior]
truncated_features = build_phase3_feature_matrix(restrict_to_window(subset_matrix, cutoff))
truncated_features = truncated_features.set_index(["product_id", "week"])

overlap = truncated_features.index
compare_cols = [
    c for c in truncated_features.columns
    if c not in ("order_demand", "product_category_name")
]
equal = np.allclose(
    global_features.loc[overlap, compare_cols].to_numpy(dtype=float),
    truncated_features[compare_cols].to_numpy(dtype=float),
    equal_nan=True,
)
assert equal, "Fold-boundary recomputation differs from global past-only pass - LEAKAGE."
print(f"Equivalence proven on {len(overlap)} overlapping product-week rows across "
      f"{len(compare_cols)} engineered columns.")

C:\Users\Hedaya_city\AppData\Local\Temp\ipykernel_35256\3412164819.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  features_full.groupby("product_id")["order_demand"].sum()


2026-09-22 00:19:28 | INFO    | src.data.target_engineering | load_demand_matrix: loaded 3469274 rows x 6 cols.


2026-09-22 00:19:28 | INFO    | src.features.feature_engineering | build_phase3_feature_matrix: starting (15900 rows).


2026-09-22 00:19:30 | INFO    | src.features.feature_engineering | add_demand_drivers: price_prev, price_chg_1w, freight_prev, category_share_prev added.


2026-09-22 00:19:30 | INFO    | src.features.feature_engineering | add_lag_features: lags (1, 2, 4) added.


...

2026-09-22 00:19:30 | INFO    | src.features.feature_engineering | add_demand_drivers: price_prev, price_chg_1w, freight_prev, category_share_prev added.


2026-09-22 00:19:30 | INFO    | src.features.feature_engineering | add_lag_features: lags (1, 2, 4) added.


2026-09-22 00:19:30 | INFO    | src.features.feature_engineering | add_rolling_features: windows (4,) aggs ('mean', 'std') added (shifted-by-1, past-only): ['rolling_demand_mean_4w', 'rolling_demand_std_4w']


2026-09-22 00:19:31 | INFO    | src.features.feature_engineering | build_phase3_feature_matrix: completed -> 8100 x 20


Equivalence proven on 8100 overlapping product-week rows across 16 engineered columns.


## 4. Chronological Split & Train-Only Imputation

The feature frame is partitioned by time (70/15/15) with the hard chronology
guard. Price/freight driver NaNs are median-imputed with statistics fit **on
train only**.

In [4]:
splits = temporal_split(features_full, "week", ratios=(0.70, 0.15, 0.15))
del features_full
gc.collect()
assert_chronological_order(splits, "week")

NUMERIC_IMPUTE = ["avg_price", "avg_freight", "price_prev", "price_chg_1w", "freight_prev"]
splits = impute_splits(splits, categorical_cols=["product_category_name"], numeric_cols=NUMERIC_IMPUTE)

for name, frame in splits.items():
    remaining = int(frame.isna().sum().sum())
    assert remaining == 0, f"{name} has {remaining} NaNs"
    LOGGER.info("%s: %d rows, %d weeks, NaN-free", name, len(frame), frame["week"].nunique())

2026-09-22 00:19:35 | INFO    | src.data.temporal_split | temporal_split: train=2428492 val=520391 test=520391 (total=3469274)


2026-09-22 00:19:45 | INFO    | src.data.imputation | impute_splits: test remaining NaNs=0


2026-09-22 00:19:45 | INFO    | src.data.imputation | impute_splits: train remaining NaNs=0


2026-09-22 00:19:45 | INFO    | src.data.imputation | impute_splits: val remaining NaNs=0


2026-09-22 00:19:46 | INFO    | notebook_03 | train: 2428492 rows, 75 weeks, NaN-free


2026-09-22 00:19:46 | INFO    | notebook_03 | val: 520391 rows, 17 weeks, NaN-free


2026-09-22 00:19:46 | INFO    | notebook_03 | test: 520391 rows, 16 weeks, NaN-free


## 5. Prepared Model Matrices

Shared design matrix for every candidate: past-only features + one-hot category
(73 levels, float32), log-domain target (`log1p`), and sub-linear sample
weights.

In [5]:
global_categories = sorted(
    pd.concat([f["product_category_name"] for f in splits.values()]).astype(str).unique()
)
prepared = {name: prepare_model_matrix(frame, categories=global_categories)
            for name, frame in splits.items()}
for name, frame in prepared.items():
    print(f"{name:>5}: {frame.shape}")

train_X, train_y, train_w = design_matrix_rows(prepared["train"])
val_X, val_y, val_w = design_matrix_rows(prepared["val"])
# Columns constant over train (unseen/sparse categories) are removed so every
# fit/predict slice shares the exact design schema (LightGBM would otherwise
# drop them fold-dependently and break the 85-vs-86 feature-count contract).
ACTIVE_COLS = [c for c in train_X.columns if train_X[c].nunique() > 1]
train_X = train_X[ACTIVE_COLS]
val_X = val_X[ACTIVE_COLS]
weeks_train = prepared["train"]["week"]
weeks_val = prepared["val"]["week"]
demand_train = prepared["train"]["order_demand"].to_numpy(dtype=float)
demand_val = prepared["val"]["order_demand"].to_numpy(dtype=float)
FEATURE_COLUMNS = list(train_X.columns)
LOGGER.info("Model design uses %d active columns.", len(FEATURE_COLUMNS))

2026-09-22 00:19:52 | INFO    | src.models.data_prep | prepare_model_matrix: 2428492 rows x 86 design cols (incl. 73 one-hot categories).


2026-09-22 00:19:54 | INFO    | src.models.data_prep | prepare_model_matrix: 520391 rows x 86 design cols (incl. 73 one-hot categories).


2026-09-22 00:19:56 | INFO    | src.models.data_prep | prepare_model_matrix: 520391 rows x 86 design cols (incl. 73 one-hot categories).


train: (2428492, 91)
  val: (520391, 91)
 test: (520391, 91)


2026-09-22 00:20:03 | INFO    | notebook_03 | Model design uses 86 active columns.


## 6. Naive Baseline (Persistence)

Benchmark: next week's demand = last observed demand per product, seeded from
training. Scored on validation with the full metric suite.

In [6]:
y_true_n, y_pred_n = naive_combined_predictions(splits["train"], splits["val"])
naive_scores = describe_metrics(y_true_n, y_pred_n)
print("Naive persistence on validation:")
for k, v in naive_scores.items():
    print(f"  {k:<22} {v:.4f}")

with mlflow.start_run(run_name="baseline_naive"):
    mlflow.log_params({"model": "naive_persistence", "scope": "validation"})
    mlflow.log_metrics(naive_scores)

Naive persistence on validation:
  wape                   1.4923
  mae                    0.0875
  rmse                   0.4376
  asymmetric_loss_2_1    0.1314


## 7. Expanding Temporal CV + Optuna Tuning (Train Only)

Tuning target: mature products (high weekly volume, stable series) so the
hyperparameter search stays fast while remaining representative. Folds are
expanding windows (`min_train=24w`, `horizon=4w`); the objective is **mean fold
validation WAPE** on the log-domain model.

In [7]:
mature_train = (
    prepared["train"].groupby("product_id")["order_demand"].sum()
    .sort_values(ascending=False).head(120).index
)
sub_prepared = prepared["train"][prepared["train"]["product_id"].isin(mature_train)].copy()
sub_X, sub_y, sub_w = design_matrix_rows(sub_prepared)
sub_X = sub_X[FEATURE_COLUMNS]
sub_weeks = sub_prepared["week"]
sub_demand = sub_prepared["order_demand"].to_numpy(dtype=float)
print(f"Tuning subset: {len(sub_prepared):,} rows | {sub_prepared['product_id'].nunique()} mature products")

sub_folds = expanding_window_splits(
    sorted(sub_weeks.unique()), n_splits=5, min_train_weeks=24, horizon=4
)
LOGGER.info("CV folds built: %d (train weeks -> val weeks)", len(sub_folds))

C:\Users\Hedaya_city\AppData\Local\Temp\ipykernel_35256\3226223692.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  prepared["train"].groupby("product_id")["order_demand"].sum()


Tuning subset: 8,899 rows | 120 mature products
2026-09-22 00:20:06 | INFO    | src.models.forecasting | expanding_window_splits: produced 5 folds.


2026-09-22 00:20:06 | INFO    | notebook_03 | CV folds built: 5 (train weeks -> val weeks)


In [8]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

studies = {}
for model_type in ["lgbm", "xgb"]:
    study = tune_hyperparameters(
        model_type, sub_X, sub_y, sub_w, sub_weeks, sub_demand, sub_folds,
        n_trials=10, seed=42,
    )
    studies[model_type] = study
    print(f"{model_type}: best mean-fold WAPE = {study.best_value:.4f}")
    print(f"  best params: {study.best_params}")

2026-09-22 00:20:08 | INFO    | src.models.forecasting | CV fold 1: WAPE=0.6579 MAE=0.24 RMSE=0.76 | train=2880 val=480


2026-09-22 00:20:09 | INFO    | src.models.forecasting | CV fold 2: WAPE=0.6361 MAE=0.43 RMSE=1.17 | train=3360 val=480


2026-09-22 00:20:11 | INFO    | src.models.forecasting | CV fold 3: WAPE=0.4981 MAE=0.39 RMSE=0.99 | train=3840 val=480


2026-09-22 00:20:12 | INFO    | src.models.forecasting | CV fold 4: WAPE=0.5377 MAE=0.62 RMSE=1.29 | train=4320 val=480


2026-09-22 00:20:14 | INFO    | src.models.forecasting | CV fold 5: WAPE=0.5714 MAE=0.76 RMSE=2.08 | train=4800 val=480


...

2026-09-22 00:22:17 | INFO    | src.models.forecasting | CV fold 3: WAPE=0.9956 MAE=0.79 RMSE=1.66 | train=3840 val=480


2026-09-22 00:22:18 | INFO    | src.models.forecasting | CV fold 4: WAPE=0.9496 MAE=1.09 RMSE=2.11 | train=4320 val=480


2026-09-22 00:22:19 | INFO    | src.models.forecasting | CV fold 5: WAPE=0.8933 MAE=1.19 RMSE=2.75 | train=4800 val=480


2026-09-22 00:22:19 | INFO    | src.models.optuna_tuning | Optuna [xgb]: best WAPE=0.5492 params={'learning_rate': 0.005670807781371429, 'max_depth': 7, 'min_child_weight': 2.460208061014161, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8059264473611898, 'reg_alpha': 0.0004982752357076451, 'reg_lambda': 0.0028888383623653178, 'n_estimators': 350}


xgb: best mean-fold WAPE = 0.5492
  best params: {'learning_rate': 0.005670807781371429, 'max_depth': 7, 'min_child_weight': 2.460208061014161, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8059264473611898, 'reg_alpha': 0.0004982752357076451, 'reg_lambda': 0.0028888383623653178, 'n_estimators': 350}


## 8. Stability Check: Best Params Across All Folds

Report per-fold WAPE for each tuned candidate on the mature-product subset.
Stable improving WAPE across folds is the convergence signal we gate on.

In [9]:
cv_reports = {}
for model_type, study in studies.items():
    params = dict(study.best_params)
    if model_type == "xgb":
        params["tree_method"] = "hist"
    model = make_model(model_type, params)
    results = run_expanding_cv(model, sub_X, sub_y, sub_w, sub_weeks, sub_demand, sub_folds)
    cv_reports[model_type] = results
    wapes = [r["wape"] for r in results]
    print(f"{model_type}: per-fold WAPE = {[f'{w:.4f}' for w in wapes]} | mean={np.mean(wapes):.4f}")

2026-09-22 00:22:19 | INFO    | src.models.forecasting | CV fold 1: WAPE=0.6579 MAE=0.24 RMSE=0.76 | train=2880 val=480


2026-09-22 00:22:20 | INFO    | src.models.forecasting | CV fold 2: WAPE=0.6361 MAE=0.43 RMSE=1.17 | train=3360 val=480


2026-09-22 00:22:21 | INFO    | src.models.forecasting | CV fold 3: WAPE=0.4981 MAE=0.39 RMSE=0.99 | train=3840 val=480


2026-09-22 00:22:22 | INFO    | src.models.forecasting | CV fold 4: WAPE=0.5377 MAE=0.62 RMSE=1.29 | train=4320 val=480


2026-09-22 00:22:24 | INFO    | src.models.forecasting | CV fold 5: WAPE=0.5714 MAE=0.76 RMSE=2.08 | train=4800 val=480


...

2026-09-22 00:22:27 | INFO    | src.models.forecasting | CV fold 2: WAPE=0.5824 MAE=0.39 RMSE=1.19 | train=3360 val=480


2026-09-22 00:22:28 | INFO    | src.models.forecasting | CV fold 3: WAPE=0.5185 MAE=0.41 RMSE=1.06 | train=3840 val=480


2026-09-22 00:22:30 | INFO    | src.models.forecasting | CV fold 4: WAPE=0.5358 MAE=0.61 RMSE=1.35 | train=4320 val=480


2026-09-22 00:22:31 | INFO    | src.models.forecasting | CV fold 5: WAPE=0.5515 MAE=0.74 RMSE=2.13 | train=4800 val=480


xgb: per-fold WAPE = ['0.5576', '0.5824', '0.5185', '0.5358', '0.5515'] | mean=0.5492


## 9. Full-Data Final Training (Early Stopping on Train Tail)

Candidates are refit on the **entire training split** with early stopping on the
last 8 training weeks (still inside the training boundary - no lookahead).
Validation is used exactly once, after training.

In [10]:
final_tune_weeks = sorted(weeks_train.unique())[-8:]
final_scores = {}
models = {}

for model_type, study in studies.items():
    params = dict(study.best_params)
    if model_type == "xgb":
        params["tree_method"] = "hist"
    run_name = f"final_{model_type}"
    with mlflow.start_run(run_name=run_name):
        model = make_model(model_type, params)
        model = fit_full_with_early_stopping(
            model, train_X, train_y, train_w, weeks_train, final_tune_weeks, early_stopping_rounds=30)
        pred_log = model.predict(val_X)
        scores = describe_metrics(demand_val, np.expm1(pred_log))
        final_scores[model_type] = scores
        models[model_type] = model
        mlflow.log_params(params)
        mlflow.log_metrics(scores)
        mlflow.log_text(json.dumps(FEATURE_COLUMNS), "feature_columns.json")
    print(f"{run_name}: " + ", ".join(f"{k}={v:.4f}" for k, v in scores.items()))

print("\nFinal comparison (validation):")
cmp = pd.DataFrame(final_scores).T
cmp.loc["naive"] = naive_scores
print(cmp.round(4).sort_values("wape").to_string())

C:\Users\Hedaya_city\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


2026-09-22 00:24:52 | INFO    | src.models.forecasting | fit_full_with_early_stopping: stopped at best_iteration=599


final_lgbm: wape=0.4001, mae=0.0235, rmse=0.2725, asymmetric_loss_2_1=0.0381


[0]	validation_0-rmse:0.15698


[1]	validation_0-rmse:0.15606


...

[347]	validation_0-rmse:0.05616


[348]	validation_0-rmse:0.05612


[349]	validation_0-rmse:0.05608


2026-09-22 00:27:51 | INFO    | src.models.forecasting | fit_full_with_early_stopping: stopped at best_iteration=349


final_xgb: wape=0.4299, mae=0.0253, rmse=0.2860, asymmetric_loss_2_1=0.0439

Final comparison (validation):
         wape     mae    rmse  asymmetric_loss_2_1
lgbm   0.4001  0.0235  0.2725               0.0381
xgb    0.4299  0.0253  0.2860               0.0439
naive  1.4923  0.0875  0.4376               0.1314


## 10. Winner Selection & MLflow PyFunc Packaging

The candidate with the lowest validation WAPE is declared winner and packaged
as a Python-function model with the exact training feature schema recorded in
its constructor. A strict input contract rejects any mistyped request.

In [11]:
winner_type = cmp["wape"].idxmin()
LOGGER.info("WINNER: %s (WAPE %.4f)", winner_type, cmp.loc[winner_type, "wape"])
assert winner_type in {"lgbm", "xgb"}, "Unexpected winner candidate."

winner_model = models[winner_type]

# Stage artifact + log PyFunc with signature.
with mlflow.start_run(run_name=f"winner_{winner_type}"):
    mlflow.log_params(studies[winner_type].best_params)
    mlflow.log_metrics(final_scores[winner_type])
    mlflow.log_text(json.dumps(FEATURE_COLUMNS), "feature_columns.json")
    logged = save_winner(winner_model, FEATURE_COLUMNS, str(ARTIFACTS_DIR))
    mlflow.log_artifact(ARTIFACTS_DIR / "model.joblib")
    winner_run_id = mlflow.active_run().info.run_id

(ARTIFACTS_DIR / "feature_columns.json").write_text(json.dumps(FEATURE_COLUMNS))
print(f"Winner '{winner_type}' packaged -> artifacts/phase3/model.joblib")
print(f"MLflow run: {winner_run_id}")

2026-09-22 00:27:53 | INFO    | notebook_03 | WINNER: lgbm (WAPE 0.4001)


2026-09-22 00:27:54 | INFO    | src.models.pyfunc_wrapper | save_winner: logging PyFunc model 'demand_forecaster'.


2026/09/22 00:27:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


C:\Users\Hedaya_city\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracing\provider.py:757: FutureWarning: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
  result = f(*args, **kwargs)


Winner 'lgbm' packaged -> artifacts/phase3/model.joblib
MLflow run: 4c9ad7d5580d41bda06154a1b3ec2649


## 11. Packaged Model Smoke Test

Load the logged PyFunc model back and verify both the prediction path and the
schema rejection path on real validation rows.

In [12]:
loaded = mlflow.pyfunc.load_model(f"runs:/{winner_run_id}/demand_forecaster")

request = prepared["val"][FEATURE_COLUMNS].iloc[:5]
out = loaded.predict(request)
assert list(out.columns) == ["pred_demand"]
assert np.all(np.isfinite(out["pred_demand"]))

try:
    loaded.predict(request.drop(columns=["price_prev"]))
    raise AssertionError("Schema validation failed to reject a missing column.")
except ValueError as exc:
    LOGGER.info("Packaged model correctly rejected malformed request: %s", exc)

print("Smoke test passed - predictions finite, malformed requests rejected.")

2026-09-22 00:28:18 | INFO    | src.models.pyfunc_wrapper | DemandForecaster.load_context: estimator loaded from C:\Users\HEDAYA~1\AppData\Local\Temp\tmpunyhnfz3\demand_forecaster\artifacts\model.joblib


2026-09-22 00:28:18 | INFO    | notebook_03 | Packaged model correctly rejected malformed request: predict: missing required columns ['price_prev']


Smoke test passed - predictions finite, malformed requests rejected.


## 12. Phase-3 Summary

The Demand Forecasting Engine is trained, tuned with expanding-window CV,
persisted through the MLflow lifecycle, and packaged behind a validated schema.
Validation metrics vs. the naive baseline demonstrate the engine's lift.

In [13]:
summary = pd.Series({
    "winner": winner_type,
    "winner_val_wape": round(float(cmp.loc[winner_type, "wape"]), 4),
    "naive_val_wape": round(float(naive_scores["wape"]), 4),
    "lgbm_val_wape": round(float(final_scores["lgbm"]["wape"]), 4),
    "xgb_val_wape": round(float(final_scores["xgb"]["wape"]), 4),
    "cv_folds": len(sub_folds),
    "optuna_trials_per_model": 10,
    "test_split_untouched": True,
    "summary": "Phase 3 complete",
})
print(summary.to_string())

winner                                 lgbm
winner_val_wape                      0.4001
naive_val_wape                       1.4923
lgbm_val_wape                        0.4001
xgb_val_wape                         0.4299
cv_folds                                  5
optuna_trials_per_model                  10
test_split_untouched                   True
summary                    Phase 3 complete
